# 02 · Train the FNO with DDP on 2× T4 (+ GPU pre-check, logs, curves, eval)

**Kaggle settings:** Accelerator → **GPU T4 ×2**, Internet → On,
Add Input → the output of `01_generate_dataset.ipynb`.

**Workflow**
1. `MODE = "precheck"` → Run All. Sweeps batch sizes with short DDP bursts and reports samples/s,
   GPU util % and peak memory per batch size, plus a recommended `BATCH_SIZE`. Nothing is trained.
2. Set `BATCH_SIZE` from the recommendation, `MODE = "train"` → Run All.

Training reuses upstream `src/` unchanged (`FNO2d`, `relative_l2`, DDP wrapping, AMP scaler,
checkpoint format), so `best.pt` drops straight into the demo app.
Outputs go to `runs/<RUN_NAME>/`: `train.log`, `metrics.csv`, `train_gpu_log.csv`, `best.pt`,
`last.pt`, `eval.json`, `per_sample_relL2.npz`, `figures/*.png`
(pre-check: `precheck.log`, `precheck.csv`, `precheck_gpu_log.csv`).

In [ ]:
import os

# ============================== CONFIG ==============================
MODE = "precheck"            # "precheck": batch-size sweep + GPU util, no training | "train"

# --- model (small preset = what the demo app serves) ---
MODES, WIDTH, LAYERS = 12, 32, 4

# --- optimisation ---
EPOCHS       = 150
BATCH_SIZE   = 32            # PER GPU. global batch = BATCH_SIZE * NPROC * GRAD_ACCUM
GRAD_ACCUM   = 1
LR           = 1e-3          # AdamW, cosine-annealed to 0 over EPOCHS
WEIGHT_DECAY = 1e-4
AMP          = True          # fp16 autocast on T4; FNO2d keeps its FFTs in fp32
SEED         = 0

# --- throughput ---
NPROC        = 2             # GPUs = DDP processes
DATA_ON_GPU  = True          # whole dataset resident on each GPU -> no DataLoader stalls

# --- evaluation / logging ---
EVAL_EVERY   = 5             # eval-mode relL2 on the FULL train set every N epochs (val: every epoch)
GPU_POLL_S   = 1.0           # nvidia-smi sampling interval

# --- pre-check ---
PRECHECK_BATCH_SIZES = [16, 32, 64, 128, 256, 512]   # per GPU; stops at the first OOM
PRECHECK_WARMUP      = 10
PRECHECK_STEPS       = 50    # minimum timed steps per batch size ...
PRECHECK_MIN_SECONDS = 8     # ... and at least this long, so the util sampler sees the burst

# --- paths ---
REPO_URL = "https://github.com/AbdullahRasheed45/fno-darcy-flow.git"  # upstream; src/ used as is
WORK_DIR = "/kaggle/working" if os.path.isdir("/kaggle/working") else os.getcwd()
DATA_DIR = None              # None = auto-find darcy_train.npz under /kaggle/input, else WORK_DIR/data
RUN_NAME = f"fno_m{MODES}_w{WIDTH}_l{LAYERS}_bs{BATCH_SIZE}x{NPROC}_e{EPOCHS}"
# ====================================================================

In [ ]:
# Setup: GPUs, upstream code, data, run_config.json
import glob, json, subprocess, sys
import torch

!nvidia-smi --query-gpu=index,name,memory.total,driver_version --format=csv
print("torch", torch.__version__, "| CUDA", torch.cuda.is_available(),
      "| GPUs", torch.cuda.device_count())
if torch.cuda.device_count() < NPROC:
    print(f"WARNING: NPROC={NPROC} but only {torch.cuda.device_count()} GPU(s) visible")

REPO_DIR = f"{WORK_DIR}/fno-darcy-flow"
if not os.path.isdir(f"{REPO_DIR}/src"):
    subprocess.run(["git", "clone", "-q", "--depth", "1", REPO_URL, REPO_DIR], check=True)
os.chdir(WORK_DIR)
os.environ["FNO_REPO"] = REPO_DIR
sys.path.insert(0, REPO_DIR)

if DATA_DIR is None:
    found = sorted(glob.glob("/kaggle/input/**/darcy_train.npz", recursive=True))
    DATA_DIR = os.path.dirname(found[0]) if found else f"{WORK_DIR}/data"
for s in ("train", "val", "test"):
    assert os.path.exists(f"{DATA_DIR}/darcy_{s}.npz"), \
        f"missing {DATA_DIR}/darcy_{s}.npz: attach the output of 01_generate_dataset.ipynb"

OUT_DIR = f"{WORK_DIR}/runs/{RUN_NAME}"
os.makedirs(f"{OUT_DIR}/figures", exist_ok=True)
CFG = {"mode": MODE, "modes": MODES, "width": WIDTH, "layers": LAYERS, "epochs": EPOCHS,
       "batch_size": BATCH_SIZE, "grad_accum": GRAD_ACCUM, "lr": LR, "weight_decay": WEIGHT_DECAY,
       "amp": AMP, "seed": SEED, "data_on_gpu": DATA_ON_GPU, "eval_every": EVAL_EVERY,
       "gpu_poll_s": GPU_POLL_S, "precheck_batch_sizes": PRECHECK_BATCH_SIZES,
       "precheck_warmup": PRECHECK_WARMUP, "precheck_steps": PRECHECK_STEPS,
       "precheck_min_seconds": PRECHECK_MIN_SECONDS, "data_dir": DATA_DIR, "out_dir": OUT_DIR}
CONFIG_PATH = f"{OUT_DIR}/run_config_{MODE}.json"
with open(CONFIG_PATH, "w") as f:
    json.dump(CFG, f, indent=2)
print("data:", DATA_DIR, "\nout: ", OUT_DIR, "\nconfig:", CONFIG_PATH)

In [ ]:
%%writefile fno_ddp_train.py
"""DDP training and GPU pre-check for the FNO Darcy surrogate.

Written by notebooks/02_train_ddp.ipynb; all settings come from the notebook's config cell
via run_config.json.

    FNO_REPO=/path/to/fno-darcy-flow torchrun --standalone --nproc_per_node=2 \
        fno_ddp_train.py --config run_config.json

mode "precheck": short timed DDP bursts over a batch-size sweep -> precheck.csv
                 (samples/s, GPU util %, peak memory, OOM). Nothing is trained or saved.
mode "train":    full DDP training -> train.log, metrics.csv, gpu_log.csv, best.pt, last.pt,
                 then eval of best.pt on train / val / test -> eval.json, per_sample_relL2.npz.

Model, parallel wrapping, AMP scaler, loss and checkpoint format are reused from upstream
src/train.py and src/infer.py, so best.pt loads in app.model.load_model() unchanged.
"""

from __future__ import annotations

import argparse
import csv
import json
import logging
import math
import os
import shutil
import subprocess
import sys
import threading
import time
from contextlib import nullcontext
from pathlib import Path

sys.path.insert(0, os.environ.get("FNO_REPO", os.getcwd()))

import numpy as np  # noqa: E402
import torch  # noqa: E402
import torch.distributed as dist  # noqa: E402

from src.infer import load_model as load_checkpoint  # noqa: E402
from src.infer import predict  # noqa: E402
from src.models.fno import FNO2d  # noqa: E402
from src.train import (gather_state_dict, make_scaler, relative_l2,  # noqa: E402
                       setup_distributed, wrap_parallel)

SPLITS = ("train", "val", "test")


# ---------------------------------------------------------------- logging / monitoring

def setup_logging(path: Path | None, rank: int) -> logging.Logger:
    """Rank 0 logs INFO to stdout and `path`; other ranks only log warnings."""
    log = logging.getLogger("fno_ddp")
    log.handlers.clear()
    log.setLevel(logging.INFO if rank == 0 else logging.WARNING)
    fmt = logging.Formatter(f"%(asctime)s [rank{rank}] %(levelname)s %(message)s", "%H:%M:%S")
    handlers: list[logging.Handler] = [logging.StreamHandler(sys.stdout)]
    if rank == 0 and path is not None:
        handlers.append(logging.FileHandler(path, mode="w"))
    for h in handlers:
        h.setFormatter(fmt)
        log.addHandler(h)
    return log


class GpuMonitor:
    """Polls nvidia-smi in a background thread and appends to a CSV.

    utilization.gpu is the % of time a kernel was running over the sample window: the
    standard "is the GPU busy" number. Disabled (no-op) without CUDA or nvidia-smi.
    """

    QUERY = "index,utilization.gpu,memory.used,memory.total"

    def __init__(self, path: Path, interval_s: float = 1.0, enabled: bool = True):
        self.enabled = (enabled and torch.cuda.is_available()
                        and shutil.which("nvidia-smi") is not None)
        self.path, self.interval = path, interval_s
        self.samples: list[tuple[float, int, float, float]] = []  # (t, gpu, util %, mem MB)
        self._lock, self._stop = threading.Lock(), threading.Event()
        self._thread: threading.Thread | None = None

    def start(self) -> "GpuMonitor":
        if self.enabled:
            with open(self.path, "w") as f:
                f.write("time_s,gpu,util_pct,mem_used_mb,mem_total_mb\n")
            self._t0 = time.time()
            self._thread = threading.Thread(target=self._loop, daemon=True)
            self._thread.start()
        return self

    def _loop(self) -> None:
        while not self._stop.is_set():
            try:
                out = subprocess.run(
                    ["nvidia-smi", f"--query-gpu={self.QUERY}", "--format=csv,noheader,nounits"],
                    capture_output=True, text=True, timeout=5).stdout
            except (subprocess.SubprocessError, OSError):
                out = ""
            now = time.time()
            rows = []
            for line in out.strip().splitlines():
                idx, util, used, total = (x.strip() for x in line.split(","))
                rows.append((now, int(idx), float(util), float(used), float(total)))
            with self._lock:
                self.samples.extend(r[:4] for r in rows)
            with open(self.path, "a") as f:
                for t, i, ut, us, to in rows:
                    f.write(f"{t - self._t0:.1f},{i},{ut:.0f},{us:.0f},{to:.0f}\n")
            self._stop.wait(self.interval)

    def mean_util(self, t0: float, t1: float) -> dict[int, float]:
        """Mean util % per GPU over samples taken in [t0, t1]."""
        with self._lock:
            window = [(g, u) for t, g, u, _ in self.samples if t0 <= t <= t1]
        out: dict[int, list[float]] = {}
        for g, u in window:
            out.setdefault(g, []).append(u)
        return {g: float(np.mean(v)) for g, v in sorted(out.items())}

    def stop(self) -> None:
        self._stop.set()
        if self._thread is not None:
            self._thread.join(timeout=10)


# ---------------------------------------------------------------- data

def load_splits(data_dir: Path, device: torch.device, splits=SPLITS):
    """Load darcy_{split}.npz, normalise with stats from the TRAIN split only.

    Same normalisation formula and stats keys as src.train.load_data, so the stats stored
    in the checkpoint are what src.infer.predict / the app expect. Tensors are moved to
    `device` once (the whole set fits easily), which removes DataLoader overhead.
    """
    raw = {s: np.load(Path(data_dir) / f"darcy_{s}.npz") for s in splits}
    a_tr = torch.from_numpy(raw["train"]["a"]).float()
    u_tr = torch.from_numpy(raw["train"]["u"]).float()
    stats = {"a_mean": float(a_tr.mean()), "a_std": float(a_tr.std()),
             "u_std": float(u_tr.std())}
    data = {}
    for s in splits:
        a = (torch.from_numpy(raw[s]["a"]).float() - stats["a_mean"]) / stats["a_std"]
        u = torch.from_numpy(raw[s]["u"]).float() / stats["u_std"]
        if device.type == "cuda":
            a, u = a.to(device), u.to(device)
        data[s] = (a, u)
    return data, stats


def rank_batches(n: int, batch_size: int, rank: int, world: int, seed: int,
                 epoch: int) -> list[torch.Tensor]:
    """This rank's index batches for one epoch.

    One shared permutation (same seed on every rank), strided across ranks and truncated
    so every rank gets the same number of samples -- DDP needs equal step counts.
    """
    perm = torch.randperm(n, generator=torch.Generator().manual_seed(seed * 100_003 + epoch))
    per_rank = n // world
    idx = perm[rank::world][:per_rank]
    return [idx[i:i + batch_size] for i in range(0, per_rank, batch_size)]


# ---------------------------------------------------------------- eval

@torch.no_grad()
def per_sample_rel_l2(net, a, u, batch_size: int, device: torch.device, amp: bool):
    """Per-sample ||pred - u|| / ||u|| (normalised space; relL2 is scale-invariant)."""
    net.eval()
    errs = []
    for i in range(0, len(a), batch_size):
        ab = a[i:i + batch_size].to(device, non_blocking=True)
        ub = u[i:i + batch_size].to(device, non_blocking=True)
        with torch.autocast(device.type, enabled=amp and device.type == "cuda"):
            pred = net(ab)
        pred = pred.float()
        errs.append(torch.linalg.vector_norm(pred - ub, dim=(-2, -1))
                    / torch.linalg.vector_norm(ub, dim=(-2, -1)))
    return torch.cat(errs) if errs else torch.zeros(0, device=device)


def dist_mean_rel_l2(net, a, u, rank, world, batch_size, device, amp) -> float:
    """Mean relL2 over the whole split, each rank evaluating a strided shard."""
    errs = per_sample_rel_l2(net, a[rank::world], u[rank::world], batch_size, device, amp)
    t = torch.tensor([float(errs.sum()), float(len(errs))], device=device)
    if world > 1:
        dist.all_reduce(t, op=dist.ReduceOp.SUM)
    return (t[0] / t[1]).item()


def summarize(errs: np.ndarray) -> dict:
    return {"n": int(len(errs)), "mean": float(errs.mean()), "median": float(np.median(errs)),
            "p95": float(np.percentile(errs, 95)), "max": float(errs.max())}


def final_eval(ckpt_path: Path, data_dir: Path, device: torch.device, batch_size: int,
               splits=SPLITS) -> tuple[dict, dict]:
    """Evaluate a saved checkpoint through the upstream inference path (raw physical units:
    src.infer.load_model + src.infer.predict), exactly as the app will use it."""
    model, stats = load_checkpoint(ckpt_path, device)
    summary, per_sample = {}, {}
    for s in splits:
        raw = np.load(Path(data_dir) / f"darcy_{s}.npz")
        a = torch.from_numpy(raw["a"]).float()
        u = torch.from_numpy(raw["u"]).float()
        pred = torch.cat([predict(model, a[i:i + batch_size], stats, device)
                          for i in range(0, len(a), batch_size)])
        errs = (torch.linalg.vector_norm(pred - u, dim=(-2, -1))
                / torch.linalg.vector_norm(u, dim=(-2, -1))).numpy()
        summary[s], per_sample[s] = summarize(errs), errs
    return summary, per_sample


# ---------------------------------------------------------------- helpers

def build_model(cfg: dict, world: int, device: torch.device):
    model = FNO2d(**model_config(cfg)).to(device)
    n_params = sum(p.numel() for p in model.parameters())
    return wrap_parallel(model, "ddp", world, device), n_params


def model_config(cfg: dict) -> dict:
    return {"modes": cfg["modes"], "width": cfg["width"], "layers": cfg["layers"]}


def peak_mem_mb(device: torch.device, world: int) -> float | None:
    """Worst-rank peak allocated memory since the last reset."""
    if device.type != "cuda":
        return None
    t = torch.tensor(float(torch.cuda.max_memory_allocated(device)), device=device)
    if world > 1:
        dist.all_reduce(t, op=dist.ReduceOp.MAX)
    return t.item() / 1e6


def sync(device: torch.device) -> None:
    if device.type == "cuda":
        torch.cuda.synchronize(device)


def train_step(model, opt, scaler, a, u, step_now: bool, accum: int, world: int,
               device: torch.device) -> torch.Tensor:
    """One micro-batch; gradient all-reduce only on the optimizer step (no_sync otherwise)."""
    sync_ctx = model.no_sync() if (world > 1 and not step_now) else nullcontext()
    with sync_ctx:
        with torch.autocast(device.type, enabled=scaler.is_enabled()):
            pred = model(a)
        loss = relative_l2(pred.float(), u)
        scaler.scale(loss / accum).backward()
    if step_now:
        scaler.step(opt)
        scaler.update()
        opt.zero_grad(set_to_none=True)
    return loss.detach()


# ---------------------------------------------------------------- modes

def run_precheck(cfg, rank, world, device, data, monitor, log, out: Path) -> list[dict]:
    """Timed DDP bursts for each batch size; returns the precheck.csv rows."""
    a_all, u_all = data["train"]
    sizes = sorted(set(cfg["precheck_batch_sizes"]) | {cfg["batch_size"]})
    warmup, steps = cfg["precheck_warmup"], cfg["precheck_steps"]  # steps = minimum
    rows = []
    for bs in sizes:
        row = {"batch_size_per_gpu": bs, "global_batch": bs * world, "status": "ok",
               "steps": "", "samples_per_s": "", "step_ms": "", "peak_mem_mb": "", "gpu_util_mean": ""}
        try:
            torch.manual_seed(cfg["seed"])
            model, _ = build_model(cfg, world, device)
            opt = torch.optim.AdamW(model.parameters(), lr=cfg["lr"])
            scaler = make_scaler("ddp", cfg["amp"], device)
            gen = torch.Generator().manual_seed(cfg["seed"] + rank)
            if device.type == "cuda":
                torch.cuda.reset_peak_memory_stats(device)

            def burst(n: int) -> None:
                for _ in range(n):
                    idx = torch.randint(len(a_all), (bs,), generator=gen)
                    a = a_all[idx].to(device, non_blocking=True)
                    u = u_all[idx].to(device, non_blocking=True)
                    train_step(model, opt, scaler, a, u, True, 1, world, device)
                sync(device)

            tw = time.time()
            burst(warmup)
            # Run long enough for the ~1 Hz nvidia-smi sampler to see the burst. All ranks
            # must run the same number of steps (DDP), so agree on the max estimate.
            est = (time.time() - tw) / max(warmup, 1)
            n = torch.tensor(float(max(steps, math.ceil(cfg["precheck_min_seconds"]
                                                         / max(est, 1e-6)))), device=device)
            if world > 1:
                dist.all_reduce(n, op=dist.ReduceOp.MAX)
            steps_run = int(min(n.item(), 20 * max(steps, 1)))
            t0 = time.time()
            burst(steps_run)
            t1 = time.time()
            dt = t1 - t0
            row.update(steps=steps_run, samples_per_s=round(bs * world * steps_run / dt, 1),
                       step_ms=round(1000 * dt / steps_run, 2))
            mem = peak_mem_mb(device, world)
            row["peak_mem_mb"] = round(mem, 1) if mem is not None else ""
            util = monitor.mean_util(t0, t1)
            for g, v in util.items():
                row[f"gpu{g}_util"] = round(v, 1)
            if util:
                row["gpu_util_mean"] = round(float(np.mean(list(util.values()))), 1)
        except torch.cuda.OutOfMemoryError:
            # Every rank runs the same batch shape, so all ranks OOM together.
            row["status"] = "oom"
        finally:
            model = opt = None
            if device.type == "cuda":
                torch.cuda.empty_cache()
        rows.append(row)
        log.info("precheck bs/gpu=%-4d global=%-5d %s  %s samples/s  %s ms/step  "
                 "peak %s MB  util %s%%", bs, bs * world, row["status"], row["samples_per_s"],
                 row["step_ms"], row["peak_mem_mb"], row["gpu_util_mean"] or "n/a")
        if row["status"] == "oom":
            break  # larger batches will not fit either

    ok = [r for r in rows if r["status"] == "ok"]
    if rank == 0:
        keys = list(dict.fromkeys(k for r in rows for k in r))
        with open(out / "precheck.csv", "w", newline="") as f:
            w = csv.DictWriter(f, fieldnames=keys)
            w.writeheader()
            w.writerows(rows)
        if ok:
            best = max(ok, key=lambda r: r["samples_per_s"])
            log.info("recommended BATCH_SIZE (per GPU) = %d: %s samples/s, util %s%%, "
                     "peak %s MB", best["batch_size_per_gpu"], best["samples_per_s"],
                     best["gpu_util_mean"] or "n/a", best["peak_mem_mb"] or "n/a")
        log.info("wrote %s", out / "precheck.csv")
    return rows


def run_train(cfg, rank, world, device, data, stats, monitor, log, out: Path) -> None:
    torch.manual_seed(cfg["seed"])
    model, n_params = build_model(cfg, world, device)
    net = model.module if world > 1 else model  # unwrapped, for eval
    opt = torch.optim.AdamW(model.parameters(), lr=cfg["lr"], weight_decay=cfg["weight_decay"])
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=cfg["epochs"])
    scaler = make_scaler("ddp", cfg["amp"], device)
    accum, bs, epochs = max(1, cfg["grad_accum"]), cfg["batch_size"], cfg["epochs"]
    a_tr, u_tr = data["train"]
    a_va, u_va = data["val"]
    eval_bs = max(bs, 64)
    per_rank = len(a_tr) // world

    log.info("model FNO2d(%s) params=%s  world=%d device=%s amp=%s  batch/gpu=%d "
             "grad_accum=%d effective_batch=%d  train/val/test=%s",
             model_config(cfg), f"{n_params:,}", world, device, scaler.is_enabled(), bs,
             accum, bs * world * accum, [len(data[s][0]) for s in SPLITS])

    n_gpus = torch.cuda.device_count() if device.type == "cuda" else 0
    fields = (["epoch", "lr", "train_loss", "val_relL2", "train_relL2_eval", "epoch_s",
               "samples_per_s", "peak_mem_mb", "gpu_util_mean"]
              + [f"gpu{g}_util" for g in range(n_gpus)])
    if rank == 0:
        with open(out / "metrics.csv", "w", newline="") as f:
            csv.writer(f).writerow(fields)

    best_val, best_epoch, t_start = math.inf, -1, time.time()
    for epoch in range(epochs):
        model.train()
        if device.type == "cuda":
            torch.cuda.reset_peak_memory_stats(device)
        batches = rank_batches(len(a_tr), bs, rank, world, cfg["seed"], epoch)
        sync(device)
        t0 = time.time()
        loss_sum = torch.zeros((), device=device)
        opt.zero_grad(set_to_none=True)
        for i, idx in enumerate(batches):
            a = a_tr[idx].to(device, non_blocking=True)
            u = u_tr[idx].to(device, non_blocking=True)
            step_now = ((i + 1) % accum == 0) or (i + 1 == len(batches))
            loss_sum += train_step(model, opt, scaler, a, u, step_now, accum, world, device)
        sync(device)
        t1 = time.time()
        epoch_s = t1 - t0

        loss = loss_sum / len(batches)
        if world > 1:
            dist.all_reduce(loss, op=dist.ReduceOp.SUM)  # ReduceOp.AVG is NCCL-only
            loss /= world
        lr = sched.get_last_lr()[0]
        sched.step()

        val = dist_mean_rel_l2(net, a_va, u_va, rank, world, eval_bs, device, cfg["amp"])
        do_train_eval = (epoch + 1) % cfg["eval_every"] == 0 or epoch + 1 == epochs
        train_eval = (dist_mean_rel_l2(net, a_tr, u_tr, rank, world, eval_bs, device,
                                       cfg["amp"]) if do_train_eval else None)
        mem = peak_mem_mb(device, world)
        util = monitor.mean_util(t0, t1)

        state = gather_state_dict(model, "ddp", world)
        if rank == 0:
            ckpt = {"model": state, "config": model_config(cfg), "stats": stats}
            if val < best_val:
                torch.save(ckpt, out / "best.pt")
            if epoch + 1 == epochs:
                torch.save(ckpt, out / "last.pt")
            row = {"epoch": epoch + 1, "lr": f"{lr:.3e}", "train_loss": f"{loss.item():.5f}",
                   "val_relL2": f"{val:.5f}",
                   "train_relL2_eval": f"{train_eval:.5f}" if train_eval is not None else "",
                   "epoch_s": f"{epoch_s:.3f}",
                   "samples_per_s": f"{per_rank * world / epoch_s:.1f}",
                   "peak_mem_mb": f"{mem:.1f}" if mem is not None else "",
                   "gpu_util_mean": f"{np.mean(list(util.values())):.1f}" if util else ""}
            row.update({f"gpu{g}_util": f"{util.get(g, float('nan')):.1f}"
                        for g in range(n_gpus)})
            with open(out / "metrics.csv", "a", newline="") as f:
                csv.DictWriter(f, fieldnames=fields).writerow(row)
            log.info("epoch %3d/%d  loss %s  val %s  train(eval) %s  lr %s  %ss  "
                     "%s samples/s  peak %s MB  util %s%%%s", epoch + 1, epochs,
                     row["train_loss"], row["val_relL2"], row["train_relL2_eval"] or "-",
                     row["lr"], row["epoch_s"], row["samples_per_s"],
                     row["peak_mem_mb"] or "n/a", row["gpu_util_mean"] or "n/a",
                     "  *best*" if val < best_val else "")
        if val < best_val:
            best_val, best_epoch = val, epoch + 1

    if world > 1:
        dist.barrier()
    if rank == 0:
        log.info("training done in %.1fs; best val relL2 %.5f at epoch %d",
                 time.time() - t_start, best_val, best_epoch)
        summary, per_sample = final_eval(out / "best.pt", Path(cfg["data_dir"]), device,
                                         eval_bs)
        for s, m in summary.items():
            log.info("eval best.pt  %-5s n=%-5d mean %.5f  median %.5f  p95 %.5f  max %.5f",
                     s, m["n"], m["mean"], m["median"], m["p95"], m["max"])
        result = {"best_epoch": best_epoch, "best_val_relL2_during_training": best_val,
                  "params": n_params, "world_size": world, "config": cfg, "splits": summary}
        (out / "eval.json").write_text(json.dumps(result, indent=2))
        np.savez(out / "per_sample_relL2.npz", **per_sample)
        log.info("wrote %s, %s", out / "eval.json", out / "per_sample_relL2.npz")


def main() -> None:
    p = argparse.ArgumentParser(description=__doc__)
    p.add_argument("--config", type=Path, required=True)
    cfg = json.loads(p.parse_args().config.read_text())

    rank, world, device = setup_distributed()
    out = Path(cfg["out_dir"])
    out.mkdir(parents=True, exist_ok=True)
    mode = cfg["mode"]
    log = setup_logging(out / f"{mode}.log", rank)
    torch.backends.cudnn.benchmark = True

    monitor = GpuMonitor(out / f"{mode}_gpu_log.csv", cfg.get("gpu_poll_s", 1.0),
                         enabled=rank == 0).start()
    if rank == 0:
        gpus = ([torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]
                if torch.cuda.is_available() else [])
        log.info("mode=%s world=%d device=%s gpus=%s torch=%s  gpu monitor=%s", mode, world,
                 device, gpus, torch.__version__, "on" if monitor.enabled else "off")
        log.info("config %s", json.dumps(cfg))
    data, stats = load_splits(Path(cfg["data_dir"]), device if cfg["data_on_gpu"]
                              else torch.device("cpu"))
    try:
        if mode == "precheck":
            run_precheck(cfg, rank, world, device, data, monitor, log, out)
        elif mode == "train":
            run_train(cfg, rank, world, device, data, stats, monitor, log, out)
        else:
            raise ValueError(f"unknown mode {mode!r}")
    finally:
        monitor.stop()
        if world > 1:
            dist.destroy_process_group()


if __name__ == "__main__":
    main()

In [ ]:
# Launch: one process per GPU. Logs stream here and are saved to OUT_DIR/<mode>.log
!torchrun --standalone --nproc_per_node={NPROC} fno_ddp_train.py --config {CONFIG_PATH}

In [ ]:
# Pre-check results: throughput, GPU utilisation and memory vs batch size.
import csv
import matplotlib.pyplot as plt
C = {"blue": "#2a78d6", "orange": "#eb6834", "aqua": "#1baf7a"}  # palette slots 1-3
pc_path = f"{OUT_DIR}/precheck.csv"
if MODE == "precheck" and os.path.exists(pc_path):
    rows = list(csv.DictReader(open(pc_path)))
    print(open(pc_path).read())
    ok = [r for r in rows if r["status"] == "ok"]
    bs = [int(r["batch_size_per_gpu"]) for r in ok]
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    panels = [("samples_per_s", "samples / s (global)", C["blue"]),
              ("gpu_util_mean", "mean GPU util %", C["orange"]),
              ("peak_mem_mb", "peak memory / GPU (MB)", C["aqua"])]
    for ax, (key, label, color) in zip(axes, panels):
        ys = [float(r[key]) if r[key] else float("nan") for r in ok]
        ax.plot(bs, ys, "-o", color=color, lw=2, ms=8)
        ax.set_xscale("log", base=2); ax.set_xticks(bs, [str(b) for b in bs])
        ax.set_xlabel("batch size per GPU"); ax.set_title(label)
        ax.grid(alpha=0.25)
    if ok:
        best = max(ok, key=lambda r: float(r["samples_per_s"]))
        axes[0].annotate(f"best: {best['batch_size_per_gpu']}",
                         (int(best["batch_size_per_gpu"]), float(best["samples_per_s"])),
                         textcoords="offset points", xytext=(0, 10), ha="center")
        print(f"-> set BATCH_SIZE = {best['batch_size_per_gpu']} and MODE = 'train'")
    fig.suptitle(f"Pre-check  FNO2d(m{MODES}, w{WIDTH}, l{LAYERS})  x{NPROC} GPU  AMP={AMP}")
    fig.tight_layout(); fig.savefig(f"{OUT_DIR}/figures/precheck.png", dpi=120); plt.show()
else:
    print("no precheck.csv (MODE != 'precheck')")

In [ ]:
# Training curves: relL2 (train loss, train eval, val), LR, throughput, GPU utilisation over time.
import csv
import numpy as np
import matplotlib.pyplot as plt
C = {"blue": "#2a78d6", "orange": "#eb6834", "aqua": "#1baf7a"}  # palette slots 1-3
m_path = f"{OUT_DIR}/metrics.csv"
if os.path.exists(m_path):
    m = list(csv.DictReader(open(m_path)))
    col = lambda k: np.array([float(r[k]) if r[k] else np.nan for r in m])
    ep = col("epoch")
    fig, axes = plt.subplots(2, 2, figsize=(14, 9))

    ax = axes[0, 0]
    ax.plot(ep, col("train_loss"), color=C["blue"], lw=2, label="train loss (train mode, running)")
    te = col("train_relL2_eval"); k = ~np.isnan(te)
    ax.plot(ep[k], te[k], "o-", color=C["aqua"], lw=2, ms=8, label="train relL2 (eval mode)")
    ax.plot(ep, col("val_relL2"), color=C["orange"], lw=2, label="val relL2")
    ax.set_yscale("log"); ax.set_xlabel("epoch"); ax.set_title("relative L2 error")
    ax.legend(); ax.grid(alpha=0.25, which="both")

    ax = axes[0, 1]
    ax.plot(ep, col("lr"), color=C["blue"], lw=2)
    ax.set_xlabel("epoch"); ax.set_title("learning rate"); ax.grid(alpha=0.25)

    ax = axes[1, 0]
    ax.plot(ep, col("samples_per_s"), color=C["blue"], lw=2)
    ax.set_xlabel("epoch"); ax.set_title("throughput, samples / s (global)"); ax.grid(alpha=0.25)

    ax = axes[1, 1]
    g_path = f"{OUT_DIR}/train_gpu_log.csv"
    if os.path.exists(g_path):
        g = list(csv.DictReader(open(g_path)))
        for gi, color in zip(sorted({r["gpu"] for r in g}), [C["blue"], C["orange"]]):
            rs = [r for r in g if r["gpu"] == gi]
            ax.plot([float(r["time_s"]) for r in rs], [float(r["util_pct"]) for r in rs],
                    color=color, lw=1.5, label=f"GPU {gi}")
        ax.legend()
    ax.set_ylim(0, 105); ax.set_xlabel("seconds"); ax.set_title("GPU utilisation %")
    ax.grid(alpha=0.25)

    fig.suptitle(f"{RUN_NAME}")
    fig.tight_layout(); fig.savefig(f"{OUT_DIR}/figures/training_curves.png", dpi=120); plt.show()
    util = col("gpu_util_mean")
    print(f"mean epoch time (excl. epoch 1): {np.mean(col('epoch_s')[1:] if len(m) > 1 else col('epoch_s')):.3f}s   "
          f"mean GPU util: {f'{np.nanmean(util):.1f}%' if np.isfinite(util).any() else 'n/a (no GPU monitor)'}")
else:
    print("no metrics.csv (MODE != 'train')")

In [ ]:
# Final evaluation of best.pt on train / val / test (through src.infer: physical units, as the app).
import json
import numpy as np
import matplotlib.pyplot as plt
import torch
from src.infer import load_model, predict
C = {"blue": "#2a78d6", "orange": "#eb6834", "aqua": "#1baf7a"}  # palette slots 1-3
e_path = f"{OUT_DIR}/eval.json"
if os.path.exists(e_path):
    ev = json.load(open(e_path))
    print(f"best epoch {ev['best_epoch']}  params {ev['params']:,}  world {ev['world_size']}")
    print(f"{'split':6s} {'n':>5s} {'mean':>8s} {'median':>8s} {'p95':>8s} {'max':>8s}")
    for s, r in ev["splits"].items():
        print(f"{s:6s} {r['n']:5d} {r['mean']:8.4f} {r['median']:8.4f} {r['p95']:8.4f} {r['max']:8.4f}")

    per = np.load(f"{OUT_DIR}/per_sample_relL2.npz")
    colors = [C["blue"], C["orange"], C["aqua"]]
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    names = list(ev["splits"])
    axes[0].bar(names, [ev["splits"][s]["mean"] for s in names], color=colors, width=0.5)
    for i, s in enumerate(names):
        axes[0].text(i, ev["splits"][s]["mean"], f"{ev['splits'][s]['mean']:.4f}",
                     ha="center", va="bottom")
    axes[0].set_title("mean relative L2 (best.pt)"); axes[0].grid(alpha=0.25, axis="y")
    bins = np.linspace(0, max(per[s].max() for s in names), 40)
    for s, color in zip(names, colors):
        axes[1].hist(per[s], bins=bins, density=True, histtype="step", lw=2, color=color, label=s)
    axes[1].set_xlabel("per-sample relL2"); axes[1].set_title("error distribution")
    axes[1].legend(); axes[1].grid(alpha=0.25)
    fig.tight_layout(); fig.savefig(f"{OUT_DIR}/figures/eval.png", dpi=120); plt.show()

    # Prediction panels: test samples at the median and worst error.
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model, stats = load_model(f"{OUT_DIR}/best.pt", device)
    test = np.load(f"{DATA_DIR}/darcy_test.npz")
    errs = per["test"]
    picks = [int(np.argsort(errs)[len(errs) // 2]), int(np.argmax(errs))]
    a = torch.from_numpy(test["a"][picks]).float()
    u = test["u"][picks]
    pred = predict(model, a, stats, device).numpy()
    fig, axes = plt.subplots(len(picks), 4, figsize=(14, 3.3 * len(picks)))
    for row, i in enumerate(picks):
        panels = [(a[row], "viridis", "a(x)"), (u[row], "magma", "true u(x)"),
                  (pred[row], "magma", "FNO u(x)"), (np.abs(pred[row] - u[row]), "inferno", "|error|")]
        for c, (img, cmap, title) in enumerate(panels):
            im = axes[row, c].imshow(img, cmap=cmap)
            fig.colorbar(im, ax=axes[row, c], fraction=0.046, pad=0.04)
            axes[row, c].set_title(title); axes[row, c].set_xticks([]); axes[row, c].set_yticks([])
        axes[row, 0].set_ylabel(f"test #{i}\nrelL2 {errs[i]:.4f}")
    fig.suptitle("test set: median (top) and worst (bottom) sample")
    fig.tight_layout(); fig.savefig(f"{OUT_DIR}/figures/test_predictions.png", dpi=120); plt.show()
else:
    print("no eval.json (MODE != 'train')")

## Use the trained model in the demo app
*Save Version*, then download `runs/<RUN_NAME>/best.pt` from the Output tab into the repo's
`checkpoints/` folder (it is in the `src.train` format `{"model", "config", "stats"}`):

```
FNO_CHECKPOINT=checkpoints/fno_small.pt uvicorn app.main:app --reload
```
The checkpoint carries its own config, so a different `MODES/WIDTH/LAYERS` also loads through
`app.model.load_model()`.